# NB_ENR_POI_COORDINATE

APPEND `lh_vv_gold.enr.gld_enr_poi_coordinate`.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:

def _keys(frame):
    return frame.select(F.col("poi_id")).where(F.col("poi_id").isNotNull()).distinct()


def build(ctx: NodeContext) -> DataFrame:
    universe = ctx.input("poi").select("poi_id")
    changed_3p = _keys(ctx.changed("poi")).union(_keys(ctx.changed("addr"))).distinct().join(universe, "poi_id", "left_semi")
    p = ctx.input("poi").join(changed_3p, "poi_id", "left_semi").alias("p")
    a = ctx.input("addr").alias("a")
    third = (p.join(a, F.col("p.poi_id") == F.col("a.poi_id"), "left").select(
        F.col("p.poi_id").alias("poi_id"),  # COL-011 — MQ-006, MQ-039, MQ-063, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("p.poi_name").alias("name"),  # COL-012 — MQ-014, MQ-063, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("a.short_address").alias("short_address"),  # COL-013 — MQ-063, docs/context/01_DATA_MODEL.md §4, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("a.ward").alias("ward"),  # COL-014 — MQ-063, docs/context/01_DATA_MODEL.md §4, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("a.district").alias("district"),  # COL-015 — MQ-063, docs/context/01_DATA_MODEL.md §4, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("a.city").alias("province_name"),  # COL-016 — MQ-063, docs/context/01_DATA_MODEL.md §4, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("a.city").alias("destination_name"),  # [SỬA 19/10] COL-017 city only
        F.col("a.lat").alias("lat"),  # COL-018 — MQ-019, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
        F.col("a.lng").alias("lng"),  # COL-019 — MQ-019, notebooks/NB_CREATE_DDL.ipynb TABLE_CONFIGS_3P
    ))
    partner_keys = _keys(ctx.changed("pn_poi")).join(ctx.input("pn_poi").select("poi_id"), "poi_id", "left_semi")
    pn = ctx.input("pn_poi").join(partner_keys, "poi_id", "left_semi")
    partner = pn.select(
        F.col("poi_id"),
        F.col("name"),
        F.col("short_address"),
        F.col("ward"),
        F.col("district"),
        F.col("province_name"),
        F.lit(None).cast("string").alias("destination_name"),
        F.lit(None).cast("double").alias("lat"),
        F.lit(None).cast("double").alias("lng"),
    )
    # COL-020 — MQ-052, MQ-058. created_at do lib gán.
    # COL-021 — MQ-039, MQ-052, MQ-058. updated_at do lib gán.
    return third.unionByName(partner)


SPEC = NodeSpec(
    trg_tbl="gld_enr_poi_coordinate",
    inputs={"poi": "slv_3p_poi", "addr": "slv_3p_poi_address", "pn_poi": "slv_pn_poi"},
    pk=("poi_id",),
    build=build,
    required_inputs=("poi",),
    write_mode="APPEND",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
